In [1]:
import pandas as pd
import torch
import numpy as np
import torch.nn as nn
#from sklearn.preprocessing import StandardScaler
#from sklearn.model_selection import LeaveOneGroupOut
from torch.utils.data import TensorDataset, DataLoader
from sklearn.preprocessing import LabelEncoder
#from sklearn.metrics import classification_report, accuracy_score, confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt
from collections import Counter
from torch.utils.data import WeightedRandomSampler
import copy
import random
from sklearn.model_selection import train_test_split
import polars as pl
from pathlib import Path
import torch.profiler
import torch.nn.functional as F

torch.set_float32_matmul_precision('high')
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

In [2]:
schema = {
    "section id": pl.Utf8,
    "recording id": pl.Utf8,

    "timestamp [ns]": pl.Int64,
    "gaze x [px]": pl.Float64,
    "gaze y [px]": pl.Float64,

    "fixation id": pl.Utf8,
    "blink id": pl.Utf8,
    "saccade id": pl.Utf8,

    "azimuth [deg]": pl.Float64,
    "elevation [deg]": pl.Float64,

    "start timestamp [ns]": pl.Int64,
    "end timestamp [ns]": pl.Int64,

    "duration [ms]": pl.Int64,
    "amplitude [px]": pl.Float64,
    "amplitude [deg]": pl.Float64,
    "mean velocity [px/s]": pl.Float64,
    "peak velocity [px/s]": pl.Float64,

    "timestamp [ms]": pl.Int64,

    "label": pl.Utf8,
    "norm_time": pl.Float64,
    "label_right": pl.Utf8,

    "duration [ms] fix": pl.Int64,
    "fixation x [px]": pl.Float64,
    "fixation y [px]": pl.Float64,

    "duration [ms] sacc": pl.Int64,
    "amplitude [deg] sacc": pl.Float64,
    "mean velocity [px/s] sacc": pl.Float64,
    "peak velocity [px/s] sacc": pl.Float64,

    "duration [ms] blink": pl.Int64,

    "gyro x [deg/s]": pl.Float64,
    "gyro y [deg/s]": pl.Float64,
    "gyro z [deg/s]": pl.Float64,

    "acceleration x [g]": pl.Float64,
    "acceleration y [g]": pl.Float64,
    "acceleration z [g]": pl.Float64,

    "roll [deg]": pl.Float64,
    "pitch [deg]": pl.Float64,
    "yaw [deg]": pl.Float64,

    "quaternion w": pl.Float64,
    "quaternion x": pl.Float64,
    "quaternion y": pl.Float64,
    "quaternion z": pl.Float64,

    "pupil diameter left [mm]": pl.Float64,
    "pupil diameter right [mm]": pl.Float64,

    "eyeball center left x [mm]": pl.Float64,
    "eyeball center left y [mm]": pl.Float64,
    "eyeball center left z [mm]": pl.Float64,

    "eyeball center right x [mm]": pl.Float64,
    "eyeball center right y [mm]": pl.Float64,
    "eyeball center right z [mm]": pl.Float64,

    "optical axis left x": pl.Float64,
    "optical axis left y": pl.Float64,
    "optical axis left z": pl.Float64,

    "optical axis right x": pl.Float64,
    "optical axis right y": pl.Float64,
    "optical axis right z": pl.Float64,

    "eyelid angle top left [rad]": pl.Float64,
    "eyelid angle bottom left [rad]": pl.Float64,
    "eyelid aperture left [mm]": pl.Float64,

    "eyelid angle top right [rad]": pl.Float64,
    "eyelid angle bottom right [rad]": pl.Float64,
    "eyelid aperture right [mm]": pl.Float64,
}

In [3]:
feature_cols = [
    "gaze x [px]",
    "gaze y [px]",
    "azimuth [deg]",
    "elevation [deg]",
    #"pupil diameter left [mm]",
    #"pupil diameter right [mm]",
    
    "optical axis left x",
    "optical axis left y",
    "optical axis left z",
    "optical axis right x",
    "optical axis right y",
    "optical axis right z",

    "gyro x [deg/s]",
    "gyro y [deg/s]",
    "gyro z [deg/s]",
    "acceleration x [g]",
    "acceleration y [g]",
    "acceleration z [g]",
    
    "roll [deg]",
    "pitch [deg]",
    "yaw [deg]",

    #"fixation id",
    #"blink id",
    #"saccade id",
    "duration [ms]",
    "fixation x [px]",
    "fixation y [px]",
    "duration [ms] sacc",
    "amplitude [deg]",
    "mean velocity [px/s]",
    "peak velocity [px/s]",
    "duration [ms] blink",
    # "quaternion w",
    # "quaternion x",
    # "quaternion y",
    # "quaternion z",
    # "eyeball center left x [mm]",
    # "eyeball center left y [mm]",
    # "eyeball center left z [mm]",
    # "eyeball center right x [mm]",
    # "eyeball center right y [mm]",
    # "eyeball center right z [mm]",
    # "eyelid angle top left [rad]",
    # "eyelid angle bottom left [rad]",
    # "eyelid aperture left [mm]",
    # "eyelid angle top right [rad]",
    # "eyelid angle bottom right [rad]",
    # "eyelid aperture right [mm]",
]

In [4]:
def load_or_create_parquet(
    parquet_path,
    csv_path,
    schema=None
):
    parquet_path = Path(parquet_path)

    if parquet_path.exists():
        print("Loading parquet...")
        return pl.scan_parquet(parquet_path)

    print("Creating parquet from CSV...")

    df = pl.read_csv(
        csv_path,
        schema_overrides=schema
    )

    df.write_parquet(parquet_path)

    df = (pl.scan_parquet(parquet_path).selectc(feature_cols))

    return df

In [5]:
data_path = '../data/'
training_data_path = data_path + "Final Training Data/"
ML_models_path = '../Models/ML/'
merged_df = load_or_create_parquet(training_data_path + "merged_output.parquet", training_data_path + "merged_output.csv")
to_label_df = pl.read_csv(data_path + "Final Data For Labeling/" + "merged_output.csv")

labels = merged_df.select("label").unique().collect().to_series().to_numpy()

Loading parquet...


In [6]:

le = LabelEncoder()
le.fit(labels)

label_map = dict(zip(le.classes_, range(len(le.classes_))))
    
merged_df = merged_df.with_columns(
    pl.col("label").replace_strict(label_map).alias("label")
)

In [7]:
class CNN1D(nn.Module):
    def __init__(self, input_channels, num_classes, activation_fn):
        super().__init__()

        self.conv = nn.Sequential(
            nn.Conv1d(input_channels, 32, 5, padding=2),
            nn.BatchNorm1d(32),
            activation_fn(),

            nn.Conv1d(32, 64, 5, padding=2),
            nn.BatchNorm1d(64),
            activation_fn(),
            nn.MaxPool1d(2),

            nn.Conv1d(64, 128, 5, padding=2),
            nn.BatchNorm1d(128),
            activation_fn(),
            nn.MaxPool1d(2),
        )

        self.fc = nn.Sequential(
            nn.Linear(128, 64),
            activation_fn(),
            nn.Dropout(0.3),
            nn.Linear(64, num_classes)
        )

    def forward(self, x):
        x = x.permute(0, 2, 1)
        x = self.conv(x)
        x = x.mean(dim=2)
        return self.fc(x)

In [8]:
def window_subject(
    df: pl.DataFrame,
    feature_cols,
    label_col,
    settings
):
    window_size = int(settings["window size"] * 1e9)
    target_length = settings["target length"]
    
    stride = int(window_size * (1 - settings["overlap"]))

    df = df.sort("timestamp [ns]")

    times = df["timestamp [ns]"].to_numpy()
    features = df.select(feature_cols).to_numpy()
    labels = df[label_col].to_numpy()

    max_time = times[-1]
    new_t = np.linspace(0, 1, target_length, dtype=np.float32)

    X, y = [], []

    start = times[0]
    
    while start + window_size <= max_time:
        end = start + window_size
    
        start_idx = np.searchsorted(times, start, side="left")
        end_idx = np.searchsorted(times, end, side="left")

        if end_idx - start_idx < settings["min samples"]:
            start += stride
            continue

        window = features[start_idx:end_idx]

        if np.isnan(window).any():
            start += stride
            continue

        label = labels[start_idx]

        window_times = times[start_idx:end_idx]
        
        denom = window_times[-1] - window_times[0]
        if denom == 0:
            start += stride
            continue

        old_t = (window_times - window_times[0]) / denom

        resampled = np.empty((target_length, window.shape[1]), dtype=np.float32)

        for i in range(window.shape[1]):
            resampled[:, i] = np.interp(new_t, old_t, window[:, i])

        X.append(resampled)
        y.append(label)

        start += stride

    return np.asarray(X, dtype=np.float32), np.asarray(y)

In [9]:
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

torch.backends.cudnn.benchmark = True

print(device)

cuda


In [10]:
def calc_feature_importance(X_test, y_test, baseline_acc, model):
    feature_importance = {}

    for feature_idx, feature_name in enumerate(feature_cols):

        X_perm = X_test.clone()

        # shuffle feature across all windows
        perm = torch.randperm(X_perm.shape[0])

        X_perm[:, :, feature_idx] = X_perm[perm, :, feature_idx]

        X_perm = X_perm.to(device)

        with torch.no_grad():

            logits = model(X_perm)

            preds = torch.argmax(logits, dim=1)

            perm_acc = (
                preds == y_test
            ).float().mean().item()

        importance = baseline_acc - perm_acc

        feature_importance[feature_name] = importance

   
    importance = sorted(
        feature_importance.items(),
        key=lambda x: x[1]
    )

    return feature_importance

In [11]:
def get_loader(settings, X_train, y_train):
    batch_size = 0
    sampler = None
    shuffle = True
    if settings["weight"] == "weighted":
            
        class_counts = np.bincount(y_train)
        weights = 1.0 / class_counts
        weights = torch.tensor(weights, dtype=torch.float32).to(device)

        loss_fn = torch.nn.CrossEntropyLoss(weight=weights)
        batch_size = 65536
    elif settings["weight"] == "random_weighted":

        loss_fn = torch.nn.CrossEntropyLoss()
        class_counts = Counter(y_train.numpy())
        
        num_samples = len(y_train)

        class_weights = {cls: num_samples / count for cls, count in class_counts.items()}

        sample_weights = [class_weights[label.item()] for label in y_train]
        sample_weights = torch.DoubleTensor(sample_weights)

        sampler = WeightedRandomSampler(
            weights=sample_weights,
            num_samples=len(sample_weights),
            replacement=True
        )
        batch_size = 3000
        shuffle = None
    else:
        loss_fn = torch.nn.CrossEntropyLoss(label_smoothing = settings["label smoothing"])
        batch_size = settings["batch size"]

    return DataLoader(
                TensorDataset(X_train, y_train),
                batch_size=batch_size,
                shuffle=shuffle,
                sampler=sampler,
                num_workers=2,
                pin_memory=True,
                prefetch_factor=2
            ), loss_fn

In [12]:
def find_avr(arr):
    accumulated = 0
    arr_len = len(arr) - 1
    for i in range(0, arr_len):
        accumulated += arr[i]

    return accumulated / arr_len    

In [13]:
# Gradient Accumulation
def train_epoch(model, loader, optimizer, loss_fn, accumulation_steps=64):
    model.train()
    optimizer.zero_grad()
    for i, (X_batch, y_batch) in enumerate(loader):
        X_batch = X_batch.to(device, non_blocking=True)
        y_batch = y_batch.to(device, non_blocking=True)

        torch.compiler.cudagraph_mark_step_begin()
        
        with torch.amp.autocast(device_type="cuda"):
            logits = model(X_batch)
            loss = loss_fn(logits, y_batch)
            loss = loss / accumulation_steps
        
        loss.backward()
        
        if (i + 1) % accumulation_steps == 0:
            optimizer.step()
            optimizer.zero_grad()

In [14]:
def validate_epoch(model, X_val, y_val):
    # --------------------
    # VALIDATION
    # --------------------
    model.eval()
    with torch.no_grad():

        X_val_device = X_val.to(device)
        y_val_device = y_val.to(device)

        logits = model(X_val_device)

        probs = torch.softmax(logits, dim=1)

        confidences, preds = torch.max(probs, dim=1)

        mask = confidences >= 0.8

        if mask.sum() == 0:

            val_metric = 0

        else:

            coverage = (
                mask.float().mean().item()
            )

            precision = (
                preds[mask] == y_val_device[mask]
            ).float().mean().item()

            val_metric = (
                precision * coverage
            )

    return val_metric

In [15]:
def find_count_above(arr, above):
    count = 0
    arr_len = len(arr) - 1
    
    for i in range(0, arr_len):
        if arr[i] >= above:
            count += 1

    return f"{count} / {arr_len + 1}"    

In [16]:
def normalize_per_subject(df: pl.DataFrame, feature_cols, group_col="recording id"):
    exprs = []

    for col in feature_cols:
        mean = pl.col(col).mean().over(group_col)
        std = pl.col(col).std().over(group_col)

        exprs.append(((pl.col(col) - mean) / (std + 1e-8)).alias(col))

    return df.with_columns(exprs)

In [17]:
def load_subject(df_lazy: pl.LazyFrame, subject_id: str) -> pl.DataFrame:
    return (
        df_lazy
        .filter(pl.col("recording id") == subject_id)
        .sort("time [ms]")
        .collect(streaming=True)
    )

In [18]:
def train_loso_ensemble(
    X_train,
    X_test,
    y_train,
    y_test,
    subject_idx,
    settings,
):
    X_train = torch.tensor(X_train, dtype=torch.float32, device=device)
    y_train = torch.tensor(y_train, dtype=torch.long, device=device)

    X_test = torch.tensor(X_test, dtype=torch.float32, device=device)
    y_test = torch.tensor(y_test, dtype=torch.long, device=device)

    # ----------------------------
    # validation split
    # ----------------------------
    X_train_np = X_train.cpu().numpy()
    y_train_np = y_train.cpu().numpy()

    X_train_main, X_val, y_train_main, y_val = train_test_split(
        X_train_np,
        y_train_np,
        test_size=0.2,
        stratify=y_train_np
    )

    X_train_main = torch.tensor(X_train_main, dtype=torch.float32)
    X_val = torch.tensor(X_val, dtype=torch.float32)

    y_train_main = torch.tensor(y_train_main, dtype=torch.long)
    y_val = torch.tensor(y_val, dtype=torch.long)

    # ----------------------------
    # ensemble
    # ----------------------------
    ensamble_settings = settings["training settings"]["ensamble settings"]
    model_settings = settings["model settings"]
    models_with_scores = []

    for ensemble_idx in range(ensamble_settings["number of trained models"]):

        print(ensemble_idx + 1)
        seed = subject_idx * 1000 + ensemble_idx
        random.seed(seed)
        np.random.seed(seed)
        torch.manual_seed(seed)

        lr_range = ensamble_settings["learning rate range"]
        wd_range = ensamble_settings["weight decay range"]

        model = CNN1D(
            input_channels=X_train.shape[-1],
            num_classes=len(torch.unique(y_train)),
            activation_fn=model_settings["activation_fn"]
        ).to(device)

        model = torch.compile(model)
        
        optimizer = torch.optim.Adam(
            model.parameters(),
            lr=np.random.uniform(lr_range[0], lr_range[1]),
            weight_decay=np.random.uniform(wd_range[0], wd_range[1])
        )

        loader, loss_fn = get_loader(ensamble_settings, X_train_main, y_train_main)

        best_val = -1
        best_state = model.state_dict()
        patience_counter = 0


        
        for epoch in range(ensamble_settings["epoch"]):
            #with torch.profiler.profile(activities=[torch.profiler.ProfilerActivity.CUDA, torch.profiler.ProfilerActivity.CPU]) as prof:
            train_epoch(model, loader, optimizer, loss_fn)
                
            #print(prof.key_averages().table(sort_by="cuda_time_total", row_limit=10))
            val_score = validate_epoch(model, X_val, y_val)

            if val_score > best_val:
                best_val = val_score
                best_state = copy.deepcopy(model.state_dict())
                patience_counter = 0
            else:
                patience_counter += 1

            if patience_counter >= ensamble_settings["patience"]:
                break
        model.load_state_dict(best_state)
        models_with_scores.append((best_val, model))

    # ----------------------------
    # TEST
    # ----------------------------
    models_with_scores.sort(key=lambda x: x[0], reverse=True)
    top_models = [m for _, m in models_with_scores[:min(ensamble_settings["number of trained models"], ensamble_settings["number of used models"])]]

    with torch.no_grad():

        probs_list = []
        for m in top_models:
            m.eval()
            prob = torch.softmax(m(X_test), dim=1).cpu()
            probs_list.append(prob)

        mean_probs = torch.mean(torch.stack(probs_list), dim=0)

        confidences, preds = torch.max(mean_probs, dim=1)

        confidences = confidences.numpy()
        preds = preds.cpu()
        y_test_np = y_test.cpu()

        results = []

        for t in settings["training settings"]["thresholds"]:
            mask = confidences >= t

            if mask.sum() == 0:
                results.append((t, 0, 0.0))
                continue

            precision = (preds[mask] == y_test_np[mask]).float().mean().item()

            results.append((t, mask.sum().item(), precision))

        acc = (preds == y_test_np).float().mean().item()

    return acc, results, len(preds)

In [19]:
merged_df.select(
    pl.col("label").unique()
).collect()

label
i64
0
1
2
3


In [ ]:
results_log = []
settings = {
    "windowing settings": {
        "window size": 0.22,
        "overlap": 0.0,
        "target length": 45, # Sample rate is 200 Hz and window size is 0.22 so 45 is around 22% of 200
        "min samples": 5
    },
    "training settings": {
        "thresholds": [0.7, 0.8, 0.9, 0.95],
        "ensamble settings": {
            "learning rate range": [0.0008, 0.0020],
            "weight decay range": [0.0002, 0.0035],
        
            "number of trained models": 8,
            "number of used models": 3,
            "patience": 5,
            "epoch": 150,
        
            "weight": None,
            "label smoothing": 0.08,
            "batch size": 256
        }
    },
    "model settings": {
        "activation_fn": nn.GELU
    }
}


subjects = (
    merged_df.select("recording id")
    .unique()
    .sort("recording id")
    .collect(engine="streaming")
    .to_series()
    .to_list()
)

subject_idx = 0
for test_subject in subjects:
    subject_idx += 1
    train_subjects = [s for s in subjects if s != test_subject]
    print(f"Working on subject number {subject_idx}: {test_subject}")
    print("Preparing data")
    # LOAD SUBJECT DATA
    train_df = normalize_per_subject(
        pl.concat([
            merged_df.filter(pl.col("recording id") == s)
            for s in train_subjects
        ]),
        feature_cols
    ).collect(engine="streaming")

    test_df = normalize_per_subject(
        merged_df.filter(pl.col("recording id") == test_subject),
        feature_cols
    ).collect(engine="streaming")

    # WINDOWING
    X_train, y_train = window_subject(train_df, feature_cols, "label", settings["windowing settings"])
    X_test, y_test = window_subject(test_df, feature_cols, "label", settings["windowing settings"])

    print_graph = False

    if print_graph:
        X_flat = X_train.reshape(len(X_train), -1)

        pca = PCA(n_components=2)
        X2 = pca.fit_transform(X_flat)

        plt.scatter(X2[:,0], X2[:,1], c=y_train, s=2)
        plt.show()

    print("Training")

    # TRAIN
    acc, results, predict_count = train_loso_ensemble(X_train, X_test, y_train, y_test, subject_idx, settings)

    results_log.append({
        "subject": test_subject,
        "acc": acc,
        "results": results,
        "predict_count": predict_count
    })
    
    print(f"acc: {acc}")
    for i in range(len(results)):
        result = results[i]
        print(f"confidence threshold (%): {result[0] * 100}, coverage: {result[1]}/{predict_count}, of which correct (%) : {result[2] * 100:.4f}")

    print("\n")

Working on subject number 1: 0fc57dc8-de68-4b41-bf69-59a5a5f2d27e
Preparing data
Training
1


In [21]:
mean_acc = sum(r["acc"] for r in results_log) / len(results_log)
print(f"mean acc: {mean_acc}")

num_thresholds = len(results_log[0]["results"])
for i in range(num_thresholds):
    threshold = results_log[0]["results"][i][0]

    mean_coverage = (
        sum(
            r["results"][i][1] / r["predict_count"]
            for r in results_log
        )
        / len(results_log)
    )
    
    mean_precision = (
        sum(r["results"][i][2] for r in results_log)
        / len(results_log)
    )

    print(
        f"threshold: {threshold * 100:.2f} %, "
        f"mean coverage: {mean_coverage*100:.2f} % "
        f"mean precision: {mean_precision*100:.4f}"
    )

print("\n")

for i, log in enumerate(results_log):
    print(f"Subject number {i+1}: {log.get('subject')}")
    print(f"acc: {log.get('acc')}")
    for result in log.get("results"):
        print(f"conf threshold (%): {result[0] * 100}, coverage: {result[1]}/{log.get('predict_count')}, correct (%) : {result[2] * 100:.4f}")

    print("\n")

mean acc: 0.5755476233634081
threshold: 70.00 %, mean coverage: 37.87 % mean precision: 70.5040
threshold: 80.00 %, mean coverage: 21.08 % mean precision: 74.9618
threshold: 90.00 %, mean coverage: 6.64 % mean precision: 80.2515
threshold: 95.00 %, mean coverage: 1.60 % mean precision: 83.6513


Subject number 1: 0fc57dc8-de68-4b41-bf69-59a5a5f2d27e
acc: 0.659111499786377
conf threshold (%): 70.0, coverage: 401/1103, correct (%) : 79.0524
conf threshold (%): 80.0, coverage: 195/1103, correct (%) : 80.5128
conf threshold (%): 90.0, coverage: 44/1103, correct (%) : 81.8182
conf threshold (%): 95.0, coverage: 4/1103, correct (%) : 50.0000


Subject number 2: 10b8d8be-39de-43e7-9260-ba4cc724d4ae
acc: 0.31200897693634033
conf threshold (%): 70.0, coverage: 389/1782, correct (%) : 37.0180
conf threshold (%): 80.0, coverage: 144/1782, correct (%) : 39.5833
conf threshold (%): 90.0, coverage: 25/1782, correct (%) : 36.0000
conf threshold (%): 95.0, coverage: 5/1782, correct (%) : 20.0000


Sub

In [22]:
del train_df, test_df, X_train, y_train, X_test, y_test

In [23]:
print(f"mean test acc: {np.mean(test_acc_arr)}")
for i in range(len(train_acc_arr) - 1):
    curr_conf = test_min_conf_pass_count_arr[i]
    print(f"train acc: {train_acc_arr[i]:.4f}, test acc: {test_acc_arr[i]:.4f}, test average confidence: {test_conf_arr[i]:.4f} \n")
    
    for j in range(len(curr_conf)):
        print(f"threshold: {curr_conf[j][0]}, coverage: {curr_conf[j][1]}, precision: {curr_conf[j][2]:.4f}")

    print("\n")

NameError: name 'test_acc_arr' is not defined

In [ ]:
folds = np.arange(1, len(train_acc) + 1)
print(test_acc)
plt.figure(figsize=(10, 5))

train_acc_proc = np.array(train_acc) * 100
test_acc_proc = np.array(test_acc) * 100
test_conf_arr_proc = np.array(test_conf_arr) * 100

plt.plot(folds, train_acc_proc, marker='o', label="Train accuracy")
plt.plot(folds, test_acc_proc, marker='o', label="Test accuracy")
plt.plot(folds, test_conf_arr_proc, marker='o', label="Avg confidence")

plt.xlabel("LOSO fold")
plt.ylabel("Score %")
plt.xticks(folds)
plt.legend()
plt.tight_layout()
plt.show()

mean_coverages = []
mean_precisions = []

for threshold_idx in range(len(thresholds)):
    
    coverages = []
    precisions = []

    for fold in test_min_conf_pass_count_arr:
        
        coverage = fold[threshold_idx][1]
        precision = fold[threshold_idx][2]

        passed, total = map(int, coverage.split("/"))
        coverage_ratio = passed / total

        coverages.append(coverage_ratio * 100)
        precisions.append(precision * 100)

    mean_coverages.append(np.mean(coverages))
    mean_precisions.append(np.mean(precisions))

plt.figure(figsize=(10, 5))

plt.plot(thresholds, mean_coverages, marker='o', label="Coverage")
plt.plot(thresholds, mean_precisions, marker='o', label="Precision")

plt.hlines(mean_coverages, thresholds[0], thresholds, linestyles="dashed", alpha=0.3)
plt.hlines(mean_precisions, thresholds[0], thresholds, linestyles="dashed", alpha=0.3)

plt.xlabel("Confidence threshold %")
plt.ylabel("Score %")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
np.unique(y, return_counts=True)